In [4]:
import pandas as pd

# 1. EXTRACT
# Load the raw data (We are using a sample of 50k rows to simulate the process quickly)
# If your computer is powerful, you can remove 'nrows=50000'
print("Extracting data...")
df = pd.read_csv('acc_16.csv', nrows=50000)

# Convert Start_Time to datetime
df['Start_Time'] = pd.to_datetime(df['Start_Time'])

# 2. TRANSFORM (Data Modeling)

# --- Create Dimension: Location ---
# We grab unique Cities and States and give them an ID
dim_location = df[['City', 'State', 'Zipcode']].drop_duplicates().reset_index(drop=True)
dim_location['Location_ID'] = dim_location.index + 1 # Create a unique ID like 1, 2, 3...

# --- Create Dimension: Weather ---
# We grab unique weather conditions
dim_weather = df[['Weather_Condition', 'Temperature(F)', 'Humidity(%)']].drop_duplicates().reset_index(drop=True)
dim_weather['Weather_ID'] = dim_weather.index + 1

# --- Create Fact Table: Accidents ---
# This is the "Master Table". It will only have IDs and Numbers.
# We merge the original data with our new Dimensions to get the IDs.

fact_table = df.merge(dim_location, on=['City', 'State', 'Zipcode'], how='left') \
               .merge(dim_weather, on=['Weather_Condition', 'Temperature(F)', 'Humidity(%)'], how='left')

# Keep only the IDs and the key metrics (Severity, Time)
fact_table = fact_table[['ID', 'Severity', 'Start_Time', 'Location_ID', 'Weather_ID', 'Distance(mi)']]

# 3. LOAD
# Save as separate tables (This mimics loading into a SQL Database)
print("Loading data to Warehouse...")
dim_location.to_csv('dim_location.csv', index=False)
dim_weather.to_csv('dim_weather.csv', index=False)
fact_table.to_csv('fact_accident.csv', index=False)

print("ETL Pipeline Completed successfully!")
print(f"Fact Table Size: {fact_table.shape}")
print(f"Location Dimension Size: {dim_location.shape}")


Extracting data...


KeyError: 'Start_Time'

In [5]:
import pandas as pd

# Load just the first 5 rows to check the column names
df_check = pd.read_csv('acc_16.csv', nrows=5)

# Print all column names as a list
print("HERE ARE YOUR ACTUAL COLUMNS:")
print(df_check.columns.tolist())


HERE ARE YOUR ACTUAL COLUMNS:
['CASENUM', 'PSU', 'PJ', 'STRATUM', 'VE_TOTAL', 'VE_FORMS', 'PVH_INVL', 'PEDS', 'PERMVIT', 'PERNOTMVIT', 'NUM_INJ', 'MONTH', 'YEAR', 'DAY_WEEK', 'HOUR', 'MINUTE', 'HARM_EV', 'ALCOHOL', 'MAX_SEV', 'MAN_COLL', 'RELJCT1', 'RELJCT2', 'TYP_INT', 'WRK_ZONE', 'REL_ROAD', 'LGT_COND', 'WEATHER1', 'WEATHER2', 'WEATHER', 'SCH_BUS', 'INT_HWY', 'CF1', 'CF2', 'CF3', 'WKDY_IM', 'HOUR_IM', 'MINUTE_IM', 'EVENT1_IM', 'MANCOL_IM', 'RELJCT1_IM', 'RELJCT2_IM', 'LGTCON_IM', 'WEATHR_IM', 'MAXSEV_IM', 'NO_INJ_IM', 'ALCHL_IM', 'URBANICITY', 'REGION', 'PSUSTRAT', 'PSU_VAR', 'WEIGHT']


In [6]:
import pandas as pd

# 1. EXTRACT
print("Extracting data from acc_16.csv...")
df = pd.read_csv('acc_16.csv')

# --- DATA CLEANING (FARS Specifics) ---
# In this dataset, '99' often means Unknown for Hour/Weather. 
# We will keep them as is for the record, but good to know.

# 2. TRANSFORM (Building the Star Schema)

# --- Create Dimension: Environment (Weather & Light) ---
# Instead of repeating "Rain/Daylight" for every crash, we make a lookup table
dim_environment = df[['WEATHER1', 'LGT_COND']].drop_duplicates().reset_index(drop=True)
dim_environment['Env_ID'] = dim_environment.index + 1  # Create unique ID

# --- Create Dimension: Time_Period ---
# We combine Year, Month, Day, Hour into a "Time Dimension"
dim_time = df[['YEAR', 'MONTH', 'DAY_WEEK', 'HOUR']].drop_duplicates().reset_index(drop=True)
dim_time['Time_ID'] = dim_time.index + 1

# --- Create Fact Table: Accident_Facts ---
# This is the Master Table.
# We merge the original data with our new Dimensions to replace text/numbers with IDs.

# Merge with Environment Dimension
fact_table = df.merge(dim_environment, on=['WEATHER1', 'LGT_COND'], how='left')

# Merge with Time Dimension
fact_table = fact_table.merge(dim_time, on=['YEAR', 'MONTH', 'DAY_WEEK', 'HOUR'], how='left')

# Select only the "Fact" columns (IDs and Measures)
# CASENUM = The Accident ID
# VE_TOTAL = Total Vehicles Involved
# NUM_INJ = Number of People Injured
# Env_ID / Time_ID = The keys to our other tables
fact_table = fact_table[['CASENUM', 'VE_TOTAL', 'NUM_INJ', 'Env_ID', 'Time_ID']]

# 3. LOAD
print("Loading data to Warehouse files...")
dim_environment.to_csv('dim_environment.csv', index=False)
dim_time.to_csv('dim_time.csv', index=False)
fact_table.to_csv('fact_crash.csv', index=False)

print("\n--- PIPELINE SUCCESSFUL ---")
print(f"Fact Table (Crashes): {fact_table.shape}")
print(f"Environment Dimension: {dim_environment.shape}")
print(f"Time Dimension: {dim_time.shape}")
print("\nCheck your folder. You should see 3 new CSV files!")


Extracting data from acc_16.csv...
Loading data to Warehouse files...

--- PIPELINE SUCCESSFUL ---
Fact Table (Crashes): (46511, 5)
Environment Dimension: (87, 3)
Time Dimension: (2061, 5)

Check your folder. You should see 3 new CSV files!
